# LearnAI Voice Assistant

This notebook implements an AI voice assistant for LearnAI institute that can handle conversations about the institute's courses, timings, fees, and other information. The assistant can handle both inbound and outbound calls using Exotel integration.

## Project Structure

1. Environment Setup
2. Configuration and Initialization (Exotel)
3. Knowledge Base Setup
4. NLP Engine Setup (Hugging Face)
5. RAG Setup
6. Voice Interface Setup (TTS and STT)
7. Phone Integration Setup (Exotel)
8. Conversation Modeling
9. Main Assistant Logic
10. Flask and ngrok Deployment

Let's start implementing each component.

## 1. Environment Setup

First, let's install all the necessary dependencies for our voice assistant.

## 2. Configuration and Initialization

Set up the Exotel configuration with SID, token, and phone number. You'll need to replace these values with your actual credentials.

In [1]:
import os
from dotenv import load_dotenv

# Create a .env file to store sensitive information
with open('.env', 'w') as f:
    f.write("EXOTEL_SID=apss61\n")
    f.write("EXOTEL_TOKEN=b4c9306d27325722c6e1e9afd9fed1d0e9cdda5f88968d0d\n")
    f.write("EXOTEL_PHONE=04045210997\n")
    f.write("NGROK_AUTH_TOKEN=2vlGzZcMzUEMK5vt9QkIFwRdWVd_6TAyT5fC46QsrrgKVE4tm\n")

# Load environment variables
load_dotenv()

# Exotel configuration
EXOTEL_SID = os.getenv('EXOTEL_SID')
EXOTEL_TOKEN = os.getenv('EXOTEL_TOKEN')
EXOTEL_PHONE = os.getenv('EXOTEL_PHONE')
NGROK_AUTH_TOKEN = os.getenv('NGROK_AUTH_TOKEN')

print("Configuration loaded successfully!")

Configuration loaded successfully!


## 3. Knowledge Base Setup

Load and process the JSON data containing information about LearnAI institute.

In [2]:
import json
import pandas as pd

# Load the JSON data
def load_knowledge_base(file_path='learnai_data.json'):
    try:
        with open(file_path, 'r') as f:
            data = json.load(f)
        return data
    except FileNotFoundError:
        print(f"File {file_path} not found. Please ensure the file exists in the current directory.")
        return None
    except json.JSONDecodeError:
        print(f"Error decoding JSON from {file_path}. Please check the file format.")
        return None

# Process the knowledge base into a format suitable for RAG
def process_knowledge_base(data):
    if not data:
        return []
    
    documents = []
    
    # Basic institute information
    basic_info = f"LearnAI is an educational institute located at {data['location']}. "
    basic_info += f"Contact information: Phone: {data['contact']['phone']}, Email: {data['contact']['email']}, "
    basic_info += f"Address: {data['contact']['address']}"
    documents.append({"content": basic_info, "type": "basic_info"})
    
    # Working hours
    hours_info = "LearnAI working hours: "
    for day, hours in data['working_hours'].items():
        hours_info += f"{day}: {hours}, "
    hours_info = hours_info[:-2]  # Remove the last comma and space
    documents.append({"content": hours_info, "type": "working_hours"})
    
    # Courses by category
    for category in data['courses']:
        cat_name = category['category']
        courses = ", ".join(category['courses'])
        course_info = f"LearnAI offers the following courses in {cat_name}: {courses}"
        documents.append({"content": course_info, "type": "courses", "category": cat_name})
    
    # Fees information
    for fee_info in data['fees']:
        fee_text = f"The {fee_info['course']} course at LearnAI has a duration of {fee_info['duration']} and costs ₹{fee_info['fee']}"
        documents.append({"content": fee_text, "type": "fees", "course": fee_info['course']})
    
    return documents

# Load and process the knowledge base
learnai_data = load_knowledge_base()
if learnai_data:
    knowledge_documents = process_knowledge_base(learnai_data)
    print(f"Knowledge base loaded with {len(knowledge_documents)} documents")
    
    # Display a sample of the processed knowledge base
    for i, doc in enumerate(knowledge_documents[:3]):
        print(f"Document {i+1}: {doc['content'][:100]}...")

Knowledge base loaded with 16 documents
Document 1: LearnAI is an educational institute located at Dilsukhnagar, Hyderabad. Contact information: Phone: ...
Document 2: LearnAI working hours: Monday: 9:00 AM – 9:00 PM, Tuesday: 9:00 AM – 9:00 PM, Wednesday: 9:00 AM – 9...
Document 3: LearnAI offers the following courses in Basic & Programming: Basic Computer, MS Office, C Programmin...


## 4. NLP Engine Setup

Set up the NLP engine using Hugging Face transformers for intent recognition and entity extraction.

In [3]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

# Define intents for our voice assistant
INTENTS = [
    "greeting",
    "course_inquiry",
    "fee_inquiry",
    "timing_inquiry",
    "location_inquiry",
    "contact_inquiry",
    "admission_process",
    "goodbye",
    "thank_you",
    "other"
]

# Sample training data for intent classification
INTENT_EXAMPLES = {
    "greeting": [
        "Hello", "Hi", "Good morning", "Good afternoon", "Hey there", 
        "Greetings", "How are you", "Nice to meet you"
    ],
    "course_inquiry": [
        "What courses do you offer", "Tell me about your courses", "Do you have Python courses",
        "I want to learn Data Science", "What AI programs do you have", "Tell me about Machine Learning courses",
        "Do you teach Digital Marketing", "What programming languages do you teach"
    ],
    "fee_inquiry": [
        "How much does it cost", "What are the fees", "How much for Data Science course",
        "Fee structure", "Course fees", "Payment options", "Is there any discount",
        "Do you offer installment plans"
    ],
    "timing_inquiry": [
        "What are your working hours", "When are you open", "Are you open on weekends",
        "What time do classes start", "Do you have evening classes", "Weekend batch timings",
        "Are you open on Sunday"
    ],
    "location_inquiry": [
        "Where are you located", "What's your address", "How to reach your institute",
        "Directions to your center", "Are you in Dilsukhnagar", "Is parking available"
    ],
    "contact_inquiry": [
        "What's your contact number", "How can I contact you", "Email address",
        "Do you have a website", "Social media handles", "WhatsApp number"
    ],
    "admission_process": [
        "How to enroll", "Admission process", "How to join", "Registration process",
        "Documents required for admission", "Can I join now", "When does the next batch start"
    ],
    "goodbye": [
        "Goodbye", "Bye", "See you later", "Have a nice day", "Good night",
        "I have to go now", "Talk to you later"
    ],
    "thank_you": [
        "Thank you", "Thanks", "Thanks for your help", "I appreciate it",
        "That was helpful", "You've been very helpful"
    ],
    "other": [
        "What's the weather today", "Who is the prime minister", "Tell me a joke",
        "What's your favorite color", "Random question", "Unrelated query"
    ]
}

# Function to create a simple intent classifier using zero-shot classification
def setup_intent_classifier():
    try:
        # Load zero-shot classification model
        classifier = pipeline("zero-shot-classification", 
                             model="facebook/bart-large-mnli")
        print("Intent classifier loaded successfully!")
        return classifier
    except Exception as e:
        print(f"Error loading intent classifier: {e}")
        return None

# Function to classify intent
def classify_intent(text, classifier):
    if not classifier:
        return "other", 0.0
    
    try:
        # Classify the text against our defined intents
        result = classifier(text, INTENTS)
        top_intent = result['labels'][0]
        confidence = result['scores'][0]
        return top_intent, confidence
    except Exception as e:
        print(f"Error classifying intent: {e}")
        return "other", 0.0

# Setup entity recognition for course names, categories, etc.
def setup_entity_recognizer():
    try:
        # Use token classification for named entity recognition
        ner = pipeline("token-classification", 
                      model="dslim/bert-base-NER")
        print("Entity recognizer loaded successfully!")
        return ner
    except Exception as e:
        print(f"Error loading entity recognizer: {e}")
        return None

# Extract entities from text
def extract_entities(text, ner):
    if not ner:
        return []
    
    try:
        entities = ner(text)
        # Group entities by word
        grouped_entities = []
        current_entity = None
        
        for entity in entities:
            if current_entity is None or entity['entity'].startswith('B-'):
                if current_entity is not None:
                    grouped_entities.append(current_entity)
                current_entity = {
                    'word': entity['word'],
                    'entity_group': entity['entity'].split('-')[1],
                    'score': entity['score']
                }
            elif entity['entity'].startswith('I-'):
                current_entity['word'] += ' ' + entity['word']
                current_entity['score'] = (current_entity['score'] + entity['score']) / 2
        
        if current_entity is not None:
            grouped_entities.append(current_entity)
            
        return grouped_entities
    except Exception as e:
        print(f"Error extracting entities: {e}")
        return []

# Initialize NLP components
intent_classifier = setup_intent_classifier()
entity_recognizer = setup_entity_recognizer()

# Test the NLP engine
if intent_classifier and entity_recognizer:
    test_text = "I want to know about your Data Science course fees"
    intent, confidence = classify_intent(test_text, intent_classifier)
    entities = extract_entities(test_text, entity_recognizer)
    
    print(f"Test text: '{test_text}'")
    print(f"Detected intent: {intent} (confidence: {confidence:.2f})")
    print(f"Extracted entities: {entities}")

C:\Users\Dell\anaconda3\envs\agentai\lib\site-packages\transformers\utils\generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


Intent classifier loaded successfully!


Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Entity recognizer loaded successfully!
Test text: 'I want to know about your Data Science course fees'
Detected intent: contact_inquiry (confidence: 0.24)
Extracted entities: [{'word': 'Data Science', 'entity_group': 'MISC', 'score': 0.9422347545623779}]


## 5. RAG Setup

Implement a Retrieval-Augmented Generation (RAG) system using Hugging Face and FAISS for efficient information retrieval.

In [4]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Setup the embedding model for RAG
def setup_embedding_model():
    try:
        # Load sentence transformer model for embeddings
        model = SentenceTransformer('all-MiniLM-L6-v2')
        print("Embedding model loaded successfully!")
        return model
    except Exception as e:
        print(f"Error loading embedding model: {e}")
        return None

# Create FAISS index for fast similarity search
def create_faiss_index(documents, embedding_model):
    if not embedding_model or not documents:
        return None, []
    
    try:
        # Extract content from documents
        texts = [doc['content'] for doc in documents]
        
        # Generate embeddings
        embeddings = embedding_model.encode(texts)
        
        # Create FAISS index
        dimension = embeddings.shape[1]
        index = faiss.IndexFlatL2(dimension)
        index.add(np.array(embeddings).astype('float32'))
        
        print(f"FAISS index created with {len(documents)} documents")
        return index, texts
    except Exception as e:
        print(f"Error creating FAISS index: {e}")
        return None, []

# Retrieve relevant documents based on query
def retrieve_documents(query, index, texts, embedding_model, k=3):
    if not index or not texts or not embedding_model:
        return []
    
    try:
        # Generate query embedding
        query_embedding = embedding_model.encode([query])
        
        # Search for similar documents
        distances, indices = index.search(np.array(query_embedding).astype('float32'), k)
        
        # Retrieve the top-k documents
        retrieved_docs = [texts[idx] for idx in indices[0]]
        return retrieved_docs
    except Exception as e:
        print(f"Error retrieving documents: {e}")
        return []

# Generate response based on retrieved documents
def generate_response(query, retrieved_docs, intent):
    if not retrieved_docs:
        return "I'm sorry, I don't have enough information to answer that question."
    
    # Combine retrieved documents into context
    context = "\n".join(retrieved_docs)
    
    # For simplicity, we'll use a template-based approach
    # In a production system, you might want to use a language model for generation
    
    # Handle different intents
    if intent == "greeting":
        return "Hello! Welcome to LearnAI. How can I assist you today?"
    
    elif intent == "goodbye":
        return "Thank you for contacting LearnAI. Have a great day!"
    
    elif intent == "thank_you":
        return "You're welcome! Is there anything else I can help you with?"
    
    elif intent == "course_inquiry":
        return f"Based on your query about courses, here's what I found: {retrieved_docs[0]}"
    
    elif intent == "fee_inquiry":
        return f"Regarding the fees, here's the information: {retrieved_docs[0]}"
    
    elif intent == "timing_inquiry":
        return f"About our timings: {retrieved_docs[0]}"
    
    elif intent == "location_inquiry":
        return f"Our location details: {retrieved_docs[0]}"
    
    elif intent == "contact_inquiry":
        return f"You can contact us at: {retrieved_docs[0]}"
    
    elif intent == "admission_process":
        return "To enroll in our courses, you can visit our institute during working hours or call us at our contact number. We'll guide you through the admission process."
    
    else:  # intent == "other"
        return f"I found this information that might help: {retrieved_docs[0]}"

# Initialize RAG components
embedding_model = setup_embedding_model()

# Create FAISS index if knowledge base is loaded
if 'knowledge_documents' in locals() and embedding_model:
    faiss_index, indexed_texts = create_faiss_index(knowledge_documents, embedding_model)
    
    # Test the RAG system
    if faiss_index:
        test_query = "What are the fees for Data Science course?"
        test_intent = "fee_inquiry"
        
        retrieved_docs = retrieve_documents(test_query, faiss_index, indexed_texts, embedding_model)
        response = generate_response(test_query, retrieved_docs, test_intent)
        
        print(f"Test query: '{test_query}'")
        print(f"Retrieved documents: {retrieved_docs}")
        print(f"Generated response: '{response}'")

Embedding model loaded successfully!
FAISS index created with 16 documents
Test query: 'What are the fees for Data Science course?'
Retrieved documents: ['The Data Science course at LearnAI has a duration of 4 months and costs ₹15000', 'The Advanced Data Science course at LearnAI has a duration of 4 months and costs ₹20000', 'The Data Analysis course at LearnAI has a duration of 2 months and costs ₹10000']
Generated response: 'Regarding the fees, here's the information: The Data Science course at LearnAI has a duration of 4 months and costs ₹15000'


## 6. Voice Interface Setup

Set up the Text-to-Speech (TTS) and Speech-to-Text (STT) components for the voice interface.

In [5]:
import speech_recognition as sr
from gtts import gTTS
import os
import tempfile
from pydub import AudioSegment
from pydub.playback import play

# Setup Speech-to-Text (STT) component
def setup_stt():
    try:
        recognizer = sr.Recognizer()
        print("Speech-to-Text component initialized successfully!")
        return recognizer
    except Exception as e:
        print(f"Error initializing Speech-to-Text component: {e}")
        return None

# Convert speech to text
def speech_to_text(audio_file_path, recognizer):
    if not recognizer:
        return ""
    
    try:
        with sr.AudioFile(audio_file_path) as source:
            audio_data = recognizer.record(source)
            text = recognizer.recognize_google(audio_data)
            return text
    except sr.UnknownValueError:
        return "Sorry, I couldn't understand the audio."
    except sr.RequestError as e:
        return f"Could not request results; {e}"
    except Exception as e:
        print(f"Error in speech-to-text conversion: {e}")
        return ""

# Setup Text-to-Speech (TTS) component
def setup_tts():
    # No specific setup needed for gTTS
    print("Text-to-Speech component initialized successfully!")
    return True

# Convert text to speech
def text_to_speech(text, output_file_path=None):
    try:
        # If no output path is provided, create a temporary file
        if not output_file_path:
            temp_dir = tempfile.gettempdir()
            output_file_path = os.path.join(temp_dir, "tts_output.mp3")
        
        # Generate speech
        tts = gTTS(text=text, lang='en', slow=False)
        tts.save(output_file_path)
        
        return output_file_path
    except Exception as e:
        print(f"Error in text-to-speech conversion: {e}")
        return None

# Initialize voice interface components
stt_recognizer = setup_stt()
tts_initialized = setup_tts()

# Test the voice interface (Note: This will only work in an environment with audio capabilities)
if stt_recognizer and tts_initialized:
    test_text = "Welcome to LearnAI. How can I help you today?"
    print(f"Testing TTS with text: '{test_text}'")
    
    # Generate speech
    audio_file = text_to_speech(test_text)
    if audio_file:
        print(f"Speech generated and saved to: {audio_file}")
        # Note: In a notebook environment, you might not be able to play audio directly
        # You would need to implement a way to play audio in your specific environment

C:\Users\Dell\anaconda3\envs\agentai\lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


Speech-to-Text component initialized successfully!
Text-to-Speech component initialized successfully!
Testing TTS with text: 'Welcome to LearnAI. How can I help you today?'
Speech generated and saved to: C:\Users\Dell\AppData\Local\Temp\tts_output.mp3


## 7. Phone Integration Setup

Set up the Exotel integration for handling inbound and outbound calls.

In [6]:
import requests
import base64
from urllib.parse import urlencode

# Exotel API class for phone integration
class ExotelAPI:
    def __init__(self, sid, token, exophone):
        self.sid = sid
        self.token = token
        self.exophone = exophone
        self.base_url = f"https://api.exotel.com/v1/Accounts/{sid}"
        self.auth = (sid, token)
    
    # Make an outbound call
    def make_call(self, to_number, callback_url):
        try:
            url = f"{self.base_url}/Calls/connect.json"
            data = {
                'From': to_number,
                'To': self.exophone,
                'CallerId': self.exophone,
                'CallType': 'trans',
                'Url': callback_url
            }
            response = requests.post(url, auth=self.auth, data=data)
            return response.json() if response.status_code == 200 else None
        except Exception as e:
            print(f"Error making outbound call: {e}")
            return None
    
    # Generate TwiML-like response for Exotel
    @staticmethod
    def generate_response_twiml(message, next_url=None):
        response = f"""<?xml version="1.0" encoding="UTF-8"?>
        <Response>
            <Say>{message}</Say>
        """
        
        if next_url:
            response += f"""    <Gather action="{next_url}" method="POST" numDigits="1">
                <Say>Press any key to continue</Say>
            </Gather>
        """
        
        response += "</Response>"
        return response
    
    # Handle incoming call
    @staticmethod
    def handle_incoming_call(callback_url):
        welcome_message = "Welcome to LearnAI. Our assistant will help you with information about our courses and services."
        return ExotelAPI.generate_response_twiml(welcome_message, callback_url)

# Initialize Exotel API
def initialize_exotel():
    try:
        exotel = ExotelAPI(
            sid=EXOTEL_SID,
            token=EXOTEL_TOKEN,
            exophone=EXOTEL_PHONE
        )
        print("Exotel API initialized successfully!")
        return exotel
    except Exception as e:
        print(f"Error initializing Exotel API: {e}")
        return None

# Initialize Exotel integration
exotel_api = initialize_exotel()

# Test Exotel integration (Note: This will not make actual calls without valid credentials)
if exotel_api:
    print("Exotel integration is ready for handling calls")
    
    # Example of generating TwiML response
    test_message = "Thank you for calling LearnAI. Our courses include Data Science, Machine Learning, and more."
    test_twiml = ExotelAPI.generate_response_twiml(test_message)
    print("Sample TwiML response:")
    print(test_twiml)

Exotel API initialized successfully!
Exotel integration is ready for handling calls
Sample TwiML response:
<?xml version="1.0" encoding="UTF-8"?>
        <Response>
            <Say>Thank you for calling LearnAI. Our courses include Data Science, Machine Learning, and more.</Say>
        </Response>


## 8. Conversation Modeling

Implement conversation flow management to handle multi-turn conversations.

In [7]:
# Conversation state management
class ConversationState:
    def __init__(self):
        self.sessions = {}
    
    # Create or get session
    def get_session(self, session_id):
        if session_id not in self.sessions:
            self.sessions[session_id] = {
                'history': [],
                'current_intent': None,
                'entities': [],
                'context': {},
                'turn_count': 0
            }
        return self.sessions[session_id]
    
    # Update session with new interaction
    def update_session(self, session_id, user_input, intent, entities, response):
        session = self.get_session(session_id)
        
        # Add to history
        session['history'].append({
            'user': user_input,
            'assistant': response,
            'turn': session['turn_count']
        })
        
        # Update session state
        session['current_intent'] = intent
        session['entities'] = entities
        session['turn_count'] += 1
        
        # Update context based on intent and entities
        if intent == "course_inquiry" and entities:
            for entity in entities:
                if entity['entity_group'] == 'ORG':
                    session['context']['course'] = entity['word']
        
        return session
    
    # End session
    def end_session(self, session_id):
        if session_id in self.sessions:
            del self.sessions[session_id]

# Conversation flow management
class ConversationManager:
    def __init__(self, intent_classifier, entity_recognizer, faiss_index, indexed_texts, embedding_model):
        self.intent_classifier = intent_classifier
        self.entity_recognizer = entity_recognizer
        self.faiss_index = faiss_index
        self.indexed_texts = indexed_texts
        self.embedding_model = embedding_model
        self.state_manager = ConversationState()
    
    # Process user input and generate response
    def process_turn(self, session_id, user_input):
        # Get or create session
        session = self.state_manager.get_session(session_id)
        
        # Classify intent
        intent, confidence = classify_intent(user_input, self.intent_classifier)
        
        # Extract entities
        entities = extract_entities(user_input, self.entity_recognizer)
        
        # Retrieve relevant documents
        retrieved_docs = retrieve_documents(
            user_input, 
            self.faiss_index, 
            self.indexed_texts, 
            self.embedding_model
        )
        
        # Generate response
        response = generate_response(user_input, retrieved_docs, intent)
        
        # Update session state
        updated_session = self.state_manager.update_session(
            session_id, 
            user_input, 
            intent, 
            entities, 
            response
        )
        
        # Check if conversation should end
        should_end = intent in ["goodbye"] or updated_session['turn_count'] > 10
        
        if should_end:
            response += "\nThank you for contacting LearnAI. Is there anything else I can help you with?"
        
        return response, should_end

# Initialize conversation manager if all components are available
if 'intent_classifier' in locals() and 'entity_recognizer' in locals() and 'faiss_index' in locals() and 'indexed_texts' in locals() and 'embedding_model' in locals():
    conversation_manager = ConversationManager(
        intent_classifier,
        entity_recognizer,
        faiss_index,
        indexed_texts,
        embedding_model
    )
    
    # Test conversation flow
    test_session_id = "test_session_123"
    test_inputs = [
        "Hello, I want to know about your courses",
        "Do you offer Data Science courses?",
        "How much does it cost?",
        "Thank you for the information",
        "Goodbye"
    ]
    
    print("Testing conversation flow:")
    for i, test_input in enumerate(test_inputs):
        print(f"\nTurn {i+1} - User: '{test_input}'")
        response, should_end = conversation_manager.process_turn(test_session_id, test_input)
        print(f"Assistant: '{response}'")
        if should_end:
            print("Conversation ended.")

Testing conversation flow:

Turn 1 - User: 'Hello, I want to know about your courses'
Assistant: 'Hello! Welcome to LearnAI. How can I assist you today?'

Turn 2 - User: 'Do you offer Data Science courses?'
Assistant: 'You can contact us at: The Advanced Data Science course at LearnAI has a duration of 4 months and costs ₹20000'

Turn 3 - User: 'How much does it cost?'
Assistant: 'I found this information that might help: The Machine Learning course at LearnAI has a duration of 3 months and costs ₹19000'

Turn 4 - User: 'Thank you for the information'
Assistant: 'You're welcome! Is there anything else I can help you with?'

Turn 5 - User: 'Goodbye'
Assistant: 'Thank you for contacting LearnAI. Have a great day!
Thank you for contacting LearnAI. Is there anything else I can help you with?'
Conversation ended.


## 9. Main Assistant Logic

Integrate all components to create the complete voice assistant.

In [8]:
# Main Voice Assistant class that integrates all components
class LearnAIVoiceAssistant:
    def __init__(self, exotel_api, conversation_manager, stt_recognizer, tts_initialized):
        self.exotel_api = exotel_api
        self.conversation_manager = conversation_manager
        self.stt_recognizer = stt_recognizer
        self.tts_initialized = tts_initialized
        self.active_calls = {}
    
    # Handle incoming call
    def handle_incoming_call(self, call_sid, from_number, callback_url):
        # Create a new session for this call
        self.active_calls[call_sid] = {
            'from_number': from_number,
            'session_id': call_sid
        }
        
        # Generate welcome message
        welcome_message = "Welcome to LearnAI. I'm your virtual assistant. How can I help you today?"
        
        # Convert to speech (in a real system, this would be streamed to the caller)
        audio_file = text_to_speech(welcome_message)
        
        # Return TwiML response
        return self.exotel_api.generate_response_twiml(welcome_message, callback_url)
    
    # Process user speech input
    def process_speech_input(self, call_sid, audio_file_path):
        if call_sid not in self.active_calls:
            return "I'm sorry, there was an error with your call. Please try again later."
        
        # Convert speech to text
        user_input = speech_to_text(audio_file_path, self.stt_recognizer)
        
        # Process the input through conversation manager
        session_id = self.active_calls[call_sid]['session_id']
        response, should_end = self.conversation_manager.process_turn(session_id, user_input)
        
        # If conversation should end, clean up
        if should_end:
            self.end_call(call_sid)
        
        return response
    
    # Make outbound call
    def make_outbound_call(self, to_number, callback_url):
        return self.exotel_api.make_call(to_number, callback_url)
    
    # End call and clean up resources
    def end_call(self, call_sid):
        if call_sid in self.active_calls:
            session_id = self.active_calls[call_sid]['session_id']
            self.conversation_manager.state_manager.end_session(session_id)
            del self.active_calls[call_sid]

# Initialize the main assistant if all components are available
if 'exotel_api' in locals() and 'conversation_manager' in locals() and 'stt_recognizer' in locals() and 'tts_initialized' in locals():
    learnai_assistant = LearnAIVoiceAssistant(
        exotel_api,
        conversation_manager,
        stt_recognizer,
        tts_initialized
    )
    
    print("LearnAI Voice Assistant initialized successfully!")
    print("The assistant is ready to handle calls through the Flask application.")

LearnAI Voice Assistant initialized successfully!
The assistant is ready to handle calls through the Flask application.


## 10. Flask and ngrok Deployment

Set up a Flask application with ngrok to expose the assistant to the internet for Exotel integration.

In [13]:
from flask import Flask, request, Response
import os
import tempfile
import subprocess
import time
import threading

# Initialize Flask app
app = Flask(__name__)

# Global variable to store the assistant instance
assistant = None
NGROK_AUTH_TOKEN = os.getenv("NGROK_AUTH_TOKEN")  # Set this in your environment

# Route for handling incoming calls
@app.route('/incoming', methods=['GET', 'POST'])
def incoming_call():
    method = request.method
    print("Received incoming call")

    if method == 'GET':
        call_sid = request.args.get('CallSid', '')
        from_number = request.args.get('From', '')
    else:
        call_sid = request.form.get('CallSid', '')
        from_number = request.form.get('From', '')

    print(f"Call from {from_number}, SID: {call_sid}")

    # ✅ REPLACE response with this
    response = """
    <?xml version="1.0" encoding="UTF-8"?>
    <Response>
        <Say voice="female">Welcome to LearnAI.</Say>
        <Gather action="/process" method="POST" input="speech dtmf" timeout="5">
            <Say voice="female">You can ask a question or press 1 to know more about our courses.</Say>
        </Gather>
        <Say voice="female">No input received. Thank you. Goodbye.</Say>
    </Response>
    """
    return Response(response, mimetype='text/xml')


# Route for processing speech during the call
@app.route('/process', methods=['POST'])
def process_call():
    if not assistant:
        return Response("Assistant not initialized", mimetype='text/plain')

    call_sid = request.form.get('CallSid', '')

    # Simulate processing speech input
    temp_dir = tempfile.gettempdir()
    dummy_audio_path = os.path.join(temp_dir, "dummy_audio.wav")

    response = "Thank you for your question. LearnAI offers various courses including Data Science, Machine Learning, and Artificial Intelligence. Our fees range from ₹10,000 to ₹22,000 depending on the course."

    callback_url = request.url_root + 'process'
    twiml_response = assistant.exotel_api.generate_response_twiml(response, callback_url)
    return Response(twiml_response, mimetype='text/xml')

# Route for initiating outbound calls
@app.route('/outbound', methods=['POST'])
def outbound_call():
    if not assistant:
        return Response("Assistant not initialized", mimetype='text/plain')

    to_number = request.form.get('to_number', '')
    if not to_number:
        return Response("Missing 'to_number' parameter", mimetype='text/plain')

    callback_url = request.url_root + 'process'
    result = assistant.make_outbound_call(to_number, callback_url)
    return Response(str(result), mimetype='application/json')

# Function to start ngrok tunnel manually
def start_ngrok():
    if NGROK_AUTH_TOKEN:
      print("Ngrok auth token already assumed to be configured.")
    ngrok_proc = subprocess.Popen(["ngrok", "http", "5000"])
    time.sleep(3)  # Allow ngrok to spin up
    # Fetch public URL using ngrok API
    import requests
    try:
        tunnel_info = requests.get("http://127.0.0.1:4040/api/tunnels").json()
        public_url = tunnel_info['tunnels'][0]['public_url']
        print(f"Public URL: {public_url}")
        print(f"Incoming Call URL: {public_url}/incoming")
        print(f"Process Call URL: {public_url}/process")
        print(f"Outbound Call URL: {public_url}/outbound")
    except Exception as e:
        print("Failed to get public URL from ngrok:", e)

# Function to start the Flask app and ngrok
def start_flask_app(assistant_instance):
    global assistant
    assistant = assistant_instance

    # Start ngrok in a background thread
    threading.Thread(target=start_ngrok, daemon=True).start()

    # Start Flask app
    app.run(host='0.0.0.0', port=5000)

# Instructions
print("To start the Flask app with ngrok v3+, run the following code in a new cell:")
print("start_flask_app(learnai_assistant)")


To start the Flask app with ngrok v3+, run the following code in a new cell:
start_flask_app(learnai_assistant)


In [14]:
start_flask_app(learnai_assistant)


Ngrok auth token already assumed to be configured.
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.20.10.4:5000
Press CTRL+C to quit


Public URL: https://c8a1-2405-201-c041-2809-e8ad-8d25-892-a09d.ngrok-free.app
Incoming Call URL: https://c8a1-2405-201-c041-2809-e8ad-8d25-892-a09d.ngrok-free.app/incoming
Process Call URL: https://c8a1-2405-201-c041-2809-e8ad-8d25-892-a09d.ngrok-free.app/process
Outbound Call URL: https://c8a1-2405-201-c041-2809-e8ad-8d25-892-a09d.ngrok-free.app/outbound


127.0.0.1 - - [03/May/2025 18:07:35] "GET /incoming?CallSid=fb4fbf93be5e31c4f93d5e2e901a1953&CallFrom=09398315821&CallTo=04045210997&Direction=incoming&Created=Sat,+03+May+2025+18:07:34&DialCallDuration=0&StartTime=2025-05-03+18:07:34&EndTime=1970-01-01+05:30:00&CallType=call-attempt&DialWhomNumber=&flow_id=947430&tenant_id=387694&From=09398315821&To=04045210997&CurrentTime=2025-05-03+18:07:35 HTTP/1.1" 200 -


Received incoming call
Call from 09398315821, SID: fb4fbf93be5e31c4f93d5e2e901a1953


127.0.0.1 - - [03/May/2025 18:07:46] "GET /incoming?CallSid=d4d5aa0e9774197525d88cd950c21953&CallFrom=09398315821&CallTo=04045210997&Direction=incoming&Created=Sat,+03+May+2025+18:07:45&DialCallDuration=0&StartTime=2025-05-03+18:07:45&EndTime=1970-01-01+05:30:00&CallType=call-attempt&DialWhomNumber=&flow_id=947430&tenant_id=387694&From=09398315821&To=04045210997&CurrentTime=2025-05-03+18:07:46 HTTP/1.1" 200 -


Received incoming call
Call from 09398315821, SID: d4d5aa0e9774197525d88cd950c21953


## Complete Setup and Usage Instructions

Here's how to use this notebook to set up and run the LearnAI Voice Assistant:

1. **Prerequisites**:
   - Exotel account with SID, token, and phone number
   - ngrok account with auth token (for public URL)
   - The LearnAI data JSON file in the same directory as this notebook

2. **Configuration**:
   - Update the `.env` file with your Exotel SID, token, and phone number
   - Add your ngrok auth token to the `.env` file

3. **Running the Assistant**:
   - Execute all cells in this notebook in order
   - Run `start_flask_app(learnai_assistant)` to start the Flask server with ngrok
   - Use the printed URLs to configure your Exotel account

4. **Exotel Configuration**:
   - In your Exotel dashboard, set up an app with the following URLs:
     - Incoming Call URL: `{ngrok_url}/incoming`
     - Process Call URL: `{ngrok_url}/process`
     - Outbound Call URL: `{ngrok_url}/outbound`

5. **Testing**:
   - Make a call to your Exotel number to test the voice assistant
   - Use the `/outbound` endpoint to initiate outbound calls

6. **Customization**:
   - Modify the knowledge base by updating the JSON file
   - Adjust the conversation flow in the `ConversationManager` class
   - Enhance the NLP capabilities by fine-tuning the models

Remember to replace the placeholder values for SID, token, and phone number with your actual Exotel credentials before running the assistant.